# Emotion Detection NLP Training Template

This notebook is a starter template for exploring data and retraining the emotion detection model.

It is not expected to run end to end until you provide a dataset with at least two columns: text and label.

## Dataset expectations

Use a CSV or TSV file with:
- one column containing raw text
- one column containing emotion labels

Example column names used below:
- `text`
- `label`

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import clean_text

## Load data

Replace the placeholder path with your own dataset file.

In [ ]:
dataset_path = PROJECT_ROOT / 'data' / 'emotion_dataset.csv'

if dataset_path.exists():
    if dataset_path.suffix.lower() in {'.tsv', '.txt'}:
        dataframe = pd.read_csv(dataset_path, sep='\t')
    else:
        dataframe = pd.read_csv(dataset_path)
    display(dataframe.head())
else:
    print(f'Dataset not found: {dataset_path}')

## Preprocess and split

This example cleans text before vectorization and encodes string labels into integers.

In [ ]:
text_column = 'text'
label_column = 'label'

if 'dataframe' in globals():
    working_frame = dataframe[[text_column, label_column]].dropna().copy()
    working_frame[text_column] = working_frame[text_column].astype(str).map(clean_text)

    label_encoder = LabelEncoder()
    encoded_labels = label_encoder.fit_transform(working_frame[label_column].astype(str))

    train_texts, test_texts, train_labels, test_labels = train_test_split(
        working_frame[text_column].tolist(),
        encoded_labels,
        test_size=0.2,
        random_state=42,
        stratify=encoded_labels if len(np.unique(encoded_labels)) > 1 else None,
    )

    print('Train size:', len(train_texts))
    print('Test size:', len(test_texts))
else:
    print('Load a dataset first.')

## Train a model

This baseline uses TF-IDF features and a linear SVM classifier.

In [ ]:
if 'train_texts' in globals():
    model_pipeline = Pipeline([
        ('vectorizer', TfidfVectorizer()),
        ('classifier', LinearSVC()),
    ])

    model_pipeline.fit(train_texts, train_labels)
    predictions = model_pipeline.predict(test_texts)

    accuracy = accuracy_score(test_labels, predictions)
    report = classification_report(test_labels, predictions, zero_division=0)

    print(f'Accuracy: {accuracy:.4f}')
    print(report)
else:
    print('Prepare the dataset split first.')

## Save artifacts

Save the trained pipeline and encoder once the model looks good.

In [ ]:
artifacts_dir = PROJECT_ROOT / 'artifacts'
artifacts_dir.mkdir(exist_ok=True)

if 'model_pipeline' in globals():
    import pickle

    with open(artifacts_dir / 'emotion_pipeline.pkl', 'wb') as f:
        pickle.dump(model_pipeline, f)

    with open(artifacts_dir / 'label_encoder.pkl', 'wb') as f:
        pickle.dump(label_encoder, f)

    print('Saved artifacts to', artifacts_dir)
else:
    print('No trained pipeline found yet.')